Choosing an angle before looking at the data would be an uninformed choice, and most likely a forced one: you decide what you want, then try to make the data support it. So the first step is to explore the API and find out what it gives us access to. Once we know that, we can see what we can actually do with it.

In [1]:
import re
import json, urllib.request, urllib.error
import duckdb
import pandas
from pydantic import BaseModel

pandas.set_option("display.max_colwidth", None)
pandas.set_option("display.max_rows", 100)

The API's help page lists every endpoint in one table. `pandas.read_html` reads every table on a web page and returns a list with one DataFrame per table. The help page has one table, so we take the first, `[0]`. In: the help page's URL. Out: a DataFrame with one row per endpoint.

In [2]:
HelpTable: pandas.DataFrame = pandas.read_html("https://api.erg.ic.ac.uk/AirQuality/help")[0]
HelpTable

,Uri,Method,Description
0,/Annual/MonitoringObjective/GroupName={GroupName},GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in XML format
1,/Annual/MonitoringObjective/GroupName={GroupName}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in JSON format
2,/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year},GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName' and 'Year. Data returned in XML format
3,/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName' and 'Year. Data returned in JSON format
4,/Annual/MonitoringObjective/LocalAuthorityId={LocalAuthorityId}/Year={Year},GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'LocalAuthorityId' and 'Year. Data returned in XML format
5,/Annual/MonitoringObjective/LocalAuthorityId={LocalAuthorityId}/Year={Year}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'LocalAuthorityId' and 'Year. Data returned in JSON format
6,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate},GET,This returns the results of annual air quality objectives for monitoring sites for SiteCode and StartDate. Data returned in XML format
7,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate}/Json,GET,This returns the results of annual air quality objectives for monitoring sites for SiteCode and StartDate. Data returned in JSON format
8,/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year},GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'SiteCode' and 'Year. Data returned in XML format
9,/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'SiteCode' and 'Year. Data returned in JSON format


Each row is one endpoint: a Uri, with the parameters it needs written in braces, an HTTP method and a description. Most endpoints appear more than once, in different formats. A Uri with no suffix returns XML, and the others end in /Json, /csv or /pdf.

To work with one endpoint at a time, we turn the table into a list of rows. `to_dict` can lay a table out in several ways, and its argument picks which one. `"records"` means one dictionary per row, with the column names as its keys (a record is another word for a row). In: a DataFrame of 70 rows. Out: a list of 70 dictionaries, like the first one below.

In [3]:
HelpRows: list[dict[str, str]] = HelpTable.to_dict("records")
print(len(HelpRows))
HelpRows[0:36]

70


[{'Uri': '/Annual/MonitoringObjective/GroupName={GroupName}',
  'Method': 'GET',
  'Description': "This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in XML format"},
 {'Uri': '/Annual/MonitoringObjective/GroupName={GroupName}/Json',
  'Method': 'GET',
  'Description': "This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in JSON format"},
 {'Uri': '/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}',
  'Method': 'GET',
  'Description': "This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName' and 'Year. Data returned in XML format"},
 {'Uri': '/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json',
  'Method': 'GET',
  'Description': "This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName' and 'Year. Data returned in JSON format"},
 {'Uri':

Written as a type, an endpoint is three pieces of text. Pydantic checks that anything built as an `Endpoint` has these three fields, each one text, and fails straight away if one is missing or isn't text. From here on, `Endpoint` (capital E) is the type, `Endpoints` is the list of them, and `endpoint` (lower case) is one of them inside a loop.

In [4]:
class Endpoint(BaseModel):
    Uri: str
    Method: str
    Description: str

Each row becomes an `Endpoint`. `**row` unpacks a dictionary into named arguments, so `Endpoint(**row)` is the same as `Endpoint(Uri=row["Uri"], Method=row["Method"], Description=row["Description"])`. In: 70 dictionaries. Out: 70 Endpoints, each one checked against the type.

In [5]:
Endpoints: list[Endpoint] = [Endpoint(**row) for row in HelpRows]
print(len(Endpoints))
Endpoints[0:]

70


[Endpoint(Uri='/Annual/MonitoringObjective/GroupName={GroupName}', Method='GET', Description="This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in XML format"),
 Endpoint(Uri='/Annual/MonitoringObjective/GroupName={GroupName}/Json', Method='GET', Description="This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in JSON format"),
 Endpoint(Uri='/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}', Method='GET', Description="This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName' and 'Year. Data returned in XML format"),
 Endpoint(Uri='/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json', Method='GET', Description="This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName' and 'Year. Data returned in JSON format"),
 Endpoint(Uri='/Annual/Monit

We keep only the JSON endpoints, because Python reads JSON straight into dictionaries and lists. In: 70 Endpoints. Out: the ones whose Uri ends in /Json.

In [6]:
JsonEndpoints: list[Endpoint] = [endpoint for endpoint in Endpoints if endpoint.Uri.endswith("/Json")]
len(JsonEndpoints)

36

`ParametersOf` maps an endpoint to the set of parameters it needs: the names in braces in its Uri. In: one Endpoint. Out: a set of names, for example {"GroupName", "Year"}.

In [7]:
def ParametersOf(endpoint: Endpoint) -> set[str]:
    return set(re.findall(r"\{(\w+)\}", endpoint.Uri))

`ParametersOf` applied to every JSON endpoint, as a table. In: 36 Endpoints. Out: a DataFrame of 36 rows, one Uri and its parameters per row. An endpoint whose parameters are set(), the empty set, needs nothing, so we can call it straight away.

In [8]:
ParametersTable: pandas.DataFrame = pandas.DataFrame(
    [(endpoint.Uri, ParametersOf(endpoint)) for endpoint in JsonEndpoints], columns=["Uri", "Parameters"])
ParametersTable

,Uri,Parameters
0,/Annual/MonitoringObjective/GroupName={GroupName}/Json,{GroupName}
1,/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json,"{GroupName, Year}"
2,/Annual/MonitoringObjective/LocalAuthorityId={LocalAuthorityId}/Year={Year}/Json,"{Year, LocalAuthorityId}"
3,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate}/Json,"{StartDate, SiteCode}"
4,/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year}/Json,"{SiteCode, Year}"
5,/Annual/MonitoringReport/SiteCode={SiteCode}/Year={Year}/Json,"{SiteCode, Year}"
6,/Annualiser/Calculate/year={year}/site={site}/backA={backA}/backB={backB}/backC={backC}/backD={backD}/species={species}/mean={mean}/Json,"{backA, backC, mean, site, year, species, backB, backD}"
7,/Annualiser/Sites/year={year}/species={species}/Json,"{species, year}"
8,/Daily/MonitoringIndex/GroupName={GroupName}/Date={Date}/Json,"{Date, GroupName}"
9,/Daily/MonitoringIndex/Latest/GroupName={GroupName}/Json,{GroupName}


Every parameter the API uses: the union of `ParametersOf` over all the JSON endpoints. These are the values we have to find before we can call every endpoint. In: 36 Endpoints. Out: one set of names.

In [9]:
AllParameters: set[str] = set().union(*[ParametersOf(endpoint) for endpoint in JsonEndpoints])
print(len(AllParameters))
AllParameters

25


{'AirQualityIndex',
 'Date',
 'EndDate',
 'GroupName',
 'LocalAuthorityId',
 'Period',
 'SiteCode',
 'SpeciesCode',
 'SpeciesName',
 'StartDate',
 'Step',
 'Units',
 'Year',
 'backA',
 'backB',
 'backC',
 'backD',
 'code',
 'limit',
 'mean',
 'montype',
 'site',
 'skip',
 'species',
 'year'}

The chain so far, each name with its shape:

```
help page URL
→ HelpTable         DataFrame: 70 rows × (Uri, Method, Description)
→ HelpRows          list of 70 dicts, one per row
→ Endpoints         list of 70 Endpoint, each one checked
→ JsonEndpoints     list of 36 Endpoint
→ ParametersTable   DataFrame: 36 rows × (Uri, Parameters)
→ AllParameters     set of 25 names

ParametersOf: Endpoint → set of names, used to make the last two
```

Now we find out what we can call. `Known` is the set of parameters we have values for, and at the start it is empty. An endpoint can be called once every parameter it needs is known, ParametersOf(endpoint) ⊆ Known, which Python writes as `<=`. In: a set of known parameter names. Out: the endpoints we can call with them.

In [10]:
def CallableWith(Known: set[str]) -> list[Endpoint]:
    return [endpoint for endpoint in JsonEndpoints if ParametersOf(endpoint) <= Known]
# return each endpoint in jsonendpoints if that endpoints parameters are a subset of known
Known: set[str] = set()
[endpoint.Uri for endpoint in CallableWith(Known)]

['/Hourly/Map/Json',
 '/Information/AirPollutionGuide/Json',
 '/Information/Groups/Json',
 '/Information/IndexHealthAdvice/Json',
 '/Information/Objective/Json',
 '/Information/Species/Json']

`Get` calls an endpoint. Each Uri in the help table is already a template: `format` fills its braces with the values we pass, so `/Information/MonitoringSites/GroupName={GroupName}/Json` with `GroupName="London"` becomes `/Information/MonitoringSites/GroupName=London/Json`. In: a Uri and a value for each of its parameters. Out: the reply, read from JSON into dictionaries and lists.

In [11]:
def Get(Uri: str, **Values: str) -> dict:
    return json.load(urllib.request.urlopen("https://api.erg.ic.ac.uk/AirQuality" + Uri.format(**Values)))

Every reply wraps its records differently: Groups in `{"Groups": {"Group": [...]}}`, Species in `{"AirQualitySpecies": {"Species": [...]}}`. Hard-coding those keys for each endpoint works, but there is a pattern behind them. The API builds its JSON from XML: every reply has one root, keys that start with @ are attributes, and the other keys are child elements. So the records are the root's one child that isn't an attribute. That's a rule, and we can write it once instead of once per endpoint.

This is a decision with several options:

```
Finding the records  { hard-code each path | a table of paths ✓ relying | a convention rule, checked ✓ exploring | search for the biggest list }
```

We use the rule to explore, because it works on endpoints we haven't seen yet, and fails loudly where the convention stops. Later, for the endpoints we rely on, we write the paths down as a table and check it. Searching for the biggest list is the option to avoid: it guesses, and a wrong guess is silent.

First, one piece of mess that comes from the XML too. XML has no lists, only repeated elements, so a list with one item comes back as a single dictionary: one record looks like `{...}`, and several look like `[{...}, {...}]`. `AsList` makes both a list. In: a dictionary or a list. Out: a list.

In [12]:
def AsList(Value: dict | list) -> list:
    return Value if isinstance(Value, list) else [Value]

`Records` is the rule. `[Root] = Reply.values()` says there is exactly one root, and raises an error if there isn't. The same goes for the one child. So the rule checks its own assumption every time, instead of guessing. In: a reply. Out: its list of records.

In [13]:
def Records(Reply: dict) -> list[dict]:
    [Root] = Reply.values()                                                          # exactly one root
    [Children] = [Value for Key, Value in Root.items() if not Key.startswith("@")]   # exactly one child that isn't an attribute
    return AsList(Children)

We call every endpoint we can, and count the records in each reply, so an empty reply would show up as 0 instead of hiding behind "ok". The help page says none of these six needs a parameter, but two of them answer 400 Bad Request. The help page is a claim about the API, and calling each endpoint is what checks it.

In [14]:
for endpoint in CallableWith(Known):
    try:
        print("ok ", len(Records(Get(endpoint.Uri))), "records", endpoint.Uri)
    except urllib.error.HTTPError as Error:
        print(Error.code, endpoint.Uri)

ok  4 records /Hourly/Map/Json
ok  38 records /Information/AirPollutionGuide/Json
ok  69 records /Information/Groups/Json
400 /Information/IndexHealthAdvice/Json


400 /Information/Objective/Json
ok  6 records /Information/Species/Json


Two more pieces of mess, shared by every endpoint. Field names start with @, so the field `@SiteCode` and the parameter `SiteCode` are two names for one thing. And a missing value is an empty string, not a null. `Clean` fixes both at the boundary: it removes the @, and turns "" into a real missing value. After this, a field and its parameter have one name. In: a table from the API. Out: the same table, with clean names and real missing values.

In [15]:
def Clean(Table: pandas.DataFrame) -> pandas.DataFrame:
    return Table.rename(columns=lambda Name: Name.removeprefix("@")).replace("", None)

The steps compose. `Get` calls the endpoint, `Records` finds its records, `json_normalize` makes them a table with one row per record, and `Clean` fixes the names and missing values. `GetTable` is that chain: GetTable = Clean ∘ json_normalize ∘ Records ∘ Get. In: a Uri and a value for each of its parameters. Out: a clean table.

In [16]:
def GetTable(Uri: str, **Values: str) -> pandas.DataFrame:
    return Clean(pandas.json_normalize(Records(Get(Uri, **Values))))

Groups and Species are the replies we can use next.

In [17]:
GroupsTable: pandas.DataFrame = GetTable("/Information/Groups/Json")
GroupsTable

,GroupName,Description,WebsiteURL
0,All,All current network LAs,http://www.erg.kcl.ac.uk/
1,AroundLondon,Selection of boroughs around London,http://www.londonair.org.uk/
2,CambridgeCity,Demo group for tender,NaN
3,Cambridgeshire,Demo group for tender,NaN
4,Essex,All 14 councils in Essex,http://www.essexair.org.uk/
5,Hampshire,Various local authorities in Hampshire,http://www.hantsair.org.uk/
6,Leicester,Leicester Council,http://www.leicester.gov.uk/
7,London,All currently active boroughs of London,http://www.londonair.org.uk/
8,LondonAirApps,A selection of boroughs in and around London to be used by various iPhone/Android/Chrome/Widget apps. Boroughs who are not in LAQN not included but London Mean is (LAID 600),NaN
9,Sussex,All 13 councils in Sussex,http://www.sussex-air.net/


In [18]:
SpeciesTable: pandas.DataFrame = GetTable("/Information/Species/Json")
SpeciesTable

,SpeciesCode,SpeciesName,Description,HealthEffect,Link
0,CO,Carbon Monoxide,"Carbon Monoxide is a colourless, odourless poisonous gas produced by incomplete, or inefficient, combustion of fuel including 'cold' or badly tuned engines.","The gas affects the transport of oxygen around the body by the blood. At very high levels, this can lead to a significant reduction in the supply of oxygen to the heart, particularly in people suffering from heart disease.",http://www.londonair.org.uk/LondonAir/guide/WhatIsCO.aspx
1,NO2,Nitrogen Dioxide,Nitrogen oxides are formed during high temperature combustion processes from the oxidation of nitrogen in the air or fuel.,"Nitrogen Dioxide has several health impacts and includes general irritation to the eyes, irritation of the respiratory system and shortness of breath.",http://www.londonair.org.uk/LondonAir/guide/WhatIsNO2.aspx
2,O3,Ozone,"Ozone is not directly emitted, but is formed by a complex set of reactions involving nitrogen oxides and hydrocarbons in the presence of sunlight.","Like nitrogen dioxide, high levels of ozone can irritate and inflame the lungs. It can also cause eye irritation, migraine and coughing.",http://www.londonair.org.uk/LondonAir/guide/WhatIsO3.aspx
3,PM10,PM10 Particulate,"Larger particles in the atmosphere are generally filtered in the nose and throat and do not cause problems, but particulate matter smaller than about 10 micrometers, referred to as PM10, can settle in the bronchi and lungs and cause health problems.","The effects of inhaling particulate matter have been widely studied in humans and animals and include asthma, respiratory problems and cardiovascular issues.",http://www.londonair.org.uk/LondonAir/guide/WhatIsPM.aspx
4,PM25,PM2.5 Particulate,"These are particles which are less than 2.5 micrometers (a thousandth of a millimetre) in diameter. Particles are made up of many different chemicals, including toxic metals and organic compounds, some of which are highly chemically reactive and researchers consider these to be involved in the toxicological effects of particles.",They are thought to have a greater effect on health than larger particles as they can be carried deeper into the lungs where they can cause inflammation and worsen heart and lung diseases.,http://www.londonair.org.uk/LondonAir/guide/WhatIsPM.aspx
5,SO2,Sulphur Dioxide,Sulphur Dioxide is produced when a material or fuel containing sulphur is burned. Pollution episodes only generally occur where there is widespread domestic use of coal or in the vicinity of coal or oil-fired power stations.,"Short-term exposure to high levels of sulphur dioxide may cause coughing, tightening of the chest and irritation of the lungs.",http://www.londonair.org.uk/LondonAir/guide/WhatIsSO2.aspx


This is how endpoints connect. A table gives values for a parameter when one of its columns has that parameter's name: the GroupName column of `GroupsTable` holds values we can pass as GroupName. Because `Clean` removed the @, that's a plain set intersection. `Reveals` is that rule. In: a table. Out: the parameter names among its columns.

In [19]:
def Reveals(Table: pandas.DataFrame) -> set[str]:
    return AllParameters & set(Table.columns)

print(Reveals(GroupsTable))
print(Reveals(SpeciesTable))

{'GroupName'}
{'SpeciesCode', 'SpeciesName'}


Known grows by what the tables reveal, and more endpoints become callable. Dates and years are different: no table reveals them, because we choose them ourselves, so we add them by hand.

In [20]:
Known = Known | Reveals(GroupsTable) | Reveals(SpeciesTable) | {"StartDate", "EndDate", "Date", "Year"}
print(Known)
[endpoint.Uri for endpoint in CallableWith(Known)]

{'StartDate', 'Year', 'Date', 'SpeciesCode', 'EndDate', 'GroupName', 'SpeciesName'}


['/Annual/MonitoringObjective/GroupName={GroupName}/Json',
 '/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json',
 '/Daily/MonitoringIndex/GroupName={GroupName}/Date={Date}/Json',
 '/Daily/MonitoringIndex/Latest/GroupName={GroupName}/Json',
 '/Hourly/Map/Json',
 '/Hourly/Map/SpeciesName={SpeciesName}/Json',
 '/Hourly/MonitoringIndex/GroupName={GroupName}/Json',
 '/Information/AirPollutionGuide/Json',
 '/Information/Groups/Json',
 '/Information/IndexHealthAdvice/Json',
 '/Information/MonitoringLocalAuthority/GroupName={GroupName}/Json',
 '/Information/MonitoringSites/GroupName={GroupName}/Json',
 '/Information/MonitoringSiteSpecies/GroupName={GroupName}/Json',
 '/Information/Objective/Json',
 '/Information/Species/Json',
 '/Information/Species/SpeciesCode={SpeciesCode}/Json']

The discovery so far:

```
Known = set()
→ CallableWith(Known)   6 endpoints, 4 of which answer
→ GetTable              GroupsTable (69 rows), SpeciesTable (6 rows)
→ Reveals               {GroupName}, {SpeciesCode, SpeciesName}
→ Known grows           + the dates we choose
→ CallableWith(Known)   16 endpoints, MonitoringSites among them

GetTable = Clean ∘ json_normalize ∘ Records ∘ Get
```

Second round. MonitoringSites is now callable, and it lists the sites in a group. `GroupsTable` describes London as "All currently active boroughs of London", so that's the group we use. In: a GroupName. Out: a table with one row per site.

In [21]:
SitesTable: pandas.DataFrame = GetTable("/Information/MonitoringSites/GroupName={GroupName}/Json", GroupName="London")
print(Reveals(SitesTable))
SitesTable

{'SiteCode'}


,LocalAuthorityCode,LocalAuthorityName,SiteCode,SiteName,SiteType,DateClosed,DateOpened,Latitude,Longitude,LatitudeWGS84,LongitudeWGS84,DisplayOffsetX,DisplayOffsetY,DataOwner,DataManager,SiteLink
0,1,Barking and Dagenham,BG3,Barking and Dagenham - North Street,Kerbside,2011-05-25 00:00:00,2007-03-16 00:00:00,51.540444,0.074418,6717454.5833,8284.17386585,0,0,Barking and Dagenham,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BG3
1,1,Barking and Dagenham,BG1,Barking and Dagenham - Rush Green,Suburban,NaN,1999-11-02 00:00:00,51.563752,0.177891,6721627.34498,19802.7355367,0,0,Barking and Dagenham,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BG1
2,1,Barking and Dagenham,BG2,Barking and Dagenham - Scrattons Farm,Suburban,NaN,1999-10-17 00:00:00,51.529389,0.132857,6715476.18683,14789.5735883,0,0,Barking and Dagenham,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BG2
3,2,Barnet,BN2,Barnet - Finchley,Urban Background,2012-04-20 00:00:00,2000-08-09 13:00:00,51.591901,-0.205992,6726669.62886,-22930.9245475,0,0,Barnet,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BN2
4,2,Barnet,BN3,Barnet - Strawberry Vale,Urban Background,2002-05-15 00:00:00,2000-08-14 14:00:00,51.6008848453589,-0.172297542087178,6728279.54795,-19180.0746501,0,0,Barnet,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BN3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,33,Westminster,WME,Westminster - Queens Park Gardens,Urban Background,2026-04-01 00:00:00,2025-07-03 00:00:00,51.5286629697555,-0.20875365618309,NaN,NaN,0,0,Westminster,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=WME
254,33,Westminster,NB1,Westminster - Strand (Northbank BID),Roadside,2023-08-12 00:00:00,2015-04-28 00:00:00,51.5119701188277,-0.116713103528546,6712364.87549212,-12992.6543679369,0,0,Westminster,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=NB1
255,33,Westminster,WM8,Westminster - Victoria,Urban Background,2016-04-04 00:00:00,2014-01-06 00:00:00,51.497066177578,-0.142438820740121,6709694.46727186,-15856.2169939846,0,0,Westminster,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=WM8
256,33,Westminster,WM9,Westminster - Victoria (Victoria BID),Roadside,2017-09-20 00:00:00,2016-08-08 00:00:00,51.4977331829735,-0.14424140916201,6709818.70995547,-16057.0573104942,0,0,Westminster,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=WM9


`SitesTable` reveals SiteCode, so Known grows again.

In [22]:
Known = Known | Reveals(SitesTable)
print(len(CallableWith(Known)))
[endpoint.Uri for endpoint in CallableWith(Known)]

26


['/Annual/MonitoringObjective/GroupName={GroupName}/Json',
 '/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json',
 '/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate}/Json',
 '/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year}/Json',
 '/Annual/MonitoringReport/SiteCode={SiteCode}/Year={Year}/Json',
 '/Daily/MonitoringIndex/GroupName={GroupName}/Date={Date}/Json',
 '/Daily/MonitoringIndex/Latest/GroupName={GroupName}/Json',
 '/Daily/MonitoringIndex/Latest/SiteCode={SiteCode}/Json',
 '/Daily/MonitoringIndex/SiteCode={SiteCode}/Date={Date}/Json',
 '/Data/Site/SiteCode={SiteCode}/StartDate={StartDate}/EndDate={EndDate}/Json',
 '/Data/SiteSpecies/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/StartDate={StartDate}/EndDate={EndDate}/Json',
 '/Data/Traffic/Site/SiteCode={SiteCode}/StartDate={StartDate}/EndDate={EndDate}/Json',
 '/Data/Wide/Site/SiteCode={SiteCode}/StartDate={StartDate}/EndDate={EndDate}/Json',
 '/Hourly/Map/Json',
 '/Hourly/Map/Speci

Data/SiteSpecies, the hourly readings, is now callable, because we know SiteCode and SpeciesCode. But knowing the names isn't knowing which values go together. Any site code with any species code makes a request that looks valid, and the API answers 200 even when the pair makes no sense, with every value empty (see Mess.md). MonitoringSiteSpecies says which pairs are real: for each site, the species it measures and the dates it started and stopped.

The pairs sit one level below the records: each site has a Species field, which `AsList` makes a list. Then `json_normalize` can make one row per pair: `record_path` names the list to turn into rows, and `meta` the site fields to copy onto each row. Which level we want is a choice about meaning, so this path is written by hand. In: the reply. Out: a table with one row per (site, species) pair.

In [23]:
SiteSpeciesSites: list[dict] = Records(Get("/Information/MonitoringSiteSpecies/GroupName={GroupName}/Json", GroupName="London"))
for site in SiteSpeciesSites:
    site["Species"] = AsList(site["Species"])

SiteSpeciesTable: pandas.DataFrame = Clean(pandas.json_normalize(SiteSpeciesSites, record_path="Species", meta=["@SiteCode"]))
SiteSpeciesTable

,SpeciesCode,SpeciesDescription,DateMeasurementStarted,DateMeasurementFinished,SiteCode
0,NO2,Nitrogen Dioxide,2008-01-01 00:00:00,2011-05-25 00:00:00,BG3
1,NO2,Nitrogen Dioxide,2008-01-01 00:00:00,NaN,BG1
2,SO2,Sulphur Dioxide,1999-10-23 00:00:00,NaN,BG1
3,NO2,Nitrogen Dioxide,2007-11-21 00:00:00,NaN,BG2
4,PM10,PM10 Particulate,1999-10-17 00:00:00,NaN,BG2
...,...,...,...,...,...
645,CO,Carbon Monoxide,2003-05-02 15:00:00,2008-04-01 00:00:00,VS1
646,NO2,Nitrogen Dioxide,2003-05-02 15:00:00,2008-04-01 00:00:00,VS1
647,O3,Ozone,2003-05-02 15:00:00,2008-04-01 00:00:00,VS1
648,PM10,PM10 Particulate,2003-05-02 15:00:00,2008-04-01 00:00:00,VS1


The valid pairs as a set: a relation between sites and species, ValidPairs ⊆ Sites × Species. `zip` pairs the two columns row by row. The table has 650 rows but fewer distinct pairs, because HS1 and HS5 are listed more than once. Most possible pairs aren't in the set, and each of those is a request that would come back empty without an error. In: `SiteSpeciesTable`. Out: a set of (SiteCode, SpeciesCode) pairs.

In [24]:
ValidPairs: set[tuple[str, str]] = set(zip(SiteSpeciesTable["SiteCode"], SiteSpeciesTable["SpeciesCode"]))
print(len(ValidPairs), "valid pairs out of", len(SitesTable) * len(SpeciesTable), "possible")

646 valid pairs out of 1548 possible


Now a decision: where do we stop a bad request? A request passes through several boundaries, and we could check at any of them. Each line below is one boundary, the braces hold its options, and ✓ marks what we chose.

```
Parameters     { nothing | format check | site ∈ SitesTable | (site, species) ∈ ValidPairs ✓ | dates within Started–Finished }
      ↓
The call       { catch everything | catch HTTPError and log it ✓ exploring | let it raise ✓ readings | retry }
      ↓
Reply content  { trust it | no values at all → raise ✓ | a threshold on % missing | a Pydantic model of the reply }
      ↓
Storage        { column types ✓ | primary key ✓ | CHECK constraints }                                   pipeline.py
      ↓
Across runs    { upsert and a changes table ✓ | a runs log ✓ | last measured hour per site ✓ }          pipeline.py
```

We need two checks, because each one catches what the other can't. Membership in `ValidPairs` stops a pair the API doesn't list, before we call. But listed doesn't mean it has data: MY1 stopped measuring PM10 in 2019, and has had no CO since 30 Oct, though CO is still listed with no end date. Only looking at the reply catches those.

What we learned, written down. The rule found the records in every reply we used. For the endpoints the pipeline relies on, we write those paths down as a table: a belief about the API's shape, kept in one place, which code can follow without guessing. In: nothing. Out: a dictionary from each Uri to the keys that lead to its records.

In [25]:
ReadingsUri: str = "/Data/SiteSpecies/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/StartDate={StartDate}/EndDate={EndDate}/Json"

RecordPaths: dict[str, list[str]] = {
    "/Information/MonitoringSites/GroupName={GroupName}/Json":       ["Sites", "Site"],
    "/Information/MonitoringSiteSpecies/GroupName={GroupName}/Json": ["Sites", "Site"],
    ReadingsUri:                                                     ["RawAQData", "Data"],
}

`Follow` walks a path down a reply, one key at a time. In: a reply and a path. Out: the records at the end of the path.

In [26]:
def Follow(Reply: dict, Path: list[str]) -> list[dict]:
    Value = Reply
    for Key in Path:
        Value = Value[Key]
    return AsList(Value)

A table of beliefs can go stale without telling us, so we check it against the API: call each endpoint with sample values, and make sure its path leads to the same records the rule finds. A key that has gone missing fails on its own, with a KeyError. `format` ignores values a Uri doesn't use, so one set of samples serves every endpoint. In: `RecordPaths`. Out: one line per endpoint, or an error naming the belief that no longer holds.

In [27]:
Samples: dict[str, str] = {"GroupName": "London", "SiteCode": "MY1", "SpeciesCode": "NO2", "StartDate": "2026-10-31", "EndDate": "2026-11-01"}

for Uri, Path in RecordPaths.items():
    Reply = Get(Uri, **Samples)
    if Follow(Reply, Path) != Records(Reply):
        raise ValueError(f"{Path} no longer leads to the records of {Uri}")
    print("holds", len(Follow(Reply, Path)), "records", Uri)

holds 258 records /Information/MonitoringSites/GroupName={GroupName}/Json


holds 254 records /Information/MonitoringSiteSpecies/GroupName={GroupName}/Json


holds 24 records /Data/SiteSpecies/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/StartDate={StartDate}/EndDate={EndDate}/Json


`GetReadings` follows the stated path, with both checks around the call. In: a SiteCode, SpeciesCode, StartDate and EndDate. Out: a clean table with one row per hour, or an error saying which check failed.

In [28]:
def GetReadings(**Values: str) -> pandas.DataFrame:
    # before the call: is the pair one the API lists?
    if (Values["SiteCode"], Values["SpeciesCode"]) not in ValidPairs:
        raise ValueError(f"Not a listed pair: {Values['SiteCode']} × {Values['SpeciesCode']}")
    Rows = Follow(Get(ReadingsUri, **Values), RecordPaths[ReadingsUri])
    # after the call: listed doesn't mean it has data
    if not any(row["@Value"] != "" for row in Rows):
        raise ValueError(f"No values: {Values['SiteCode']} × {Values['SpeciesCode']}, {Values['StartDate']} to {Values['EndDate']}")
    return Clean(pandas.json_normalize(Rows)).assign(**Values)

Five requests for the same day, one for each kind of outcome: a real pair with data, a site that doesn't exist, a species the site doesn't measure, a species it stopped measuring in 2019, and one it still lists but stopped on 30 Oct (true on 1 Nov 2026).

In [29]:
for site, species in [("MY1", "NO2"), ("ZZZ", "NO2"), ("GT1", "CO"), ("MY1", "PM10"), ("MY1", "CO")]:
    try:
        print("ok", site, species, len(GetReadings(SiteCode=site, SpeciesCode=species, StartDate="2026-10-31", EndDate="2026-11-01")), "hours")
    except ValueError as Error:
        print(Error)

ok MY1 NO2 24 hours
Not a listed pair: ZZZ × NO2
Not a listed pair: GT1 × CO
No values: MY1 × PM10, 2026-10-31 to 2026-11-01


No values: MY1 × CO, 2026-10-31 to 2026-11-01


Where each piece of mess lives. Variation that differs from endpoint to endpoint sits in a table. Variation with a pattern sits in a rule, which is a table compressed into one line. Either way it has one home, at the boundary, and the code after it is the same for every endpoint. Everything we found is in Mess.md.

```
Mess                                                    Where it lives
each endpoint wraps its records under different keys    Records, a rule
the paths of the endpoints we rely on                   RecordPaths, a table, checked against the API
a list with one item comes back as a dictionary         AsList, a rule
field names start with @, parameters don't              Clean, a rule
a missing value is ""                                   Clean, a rule
any site × species looks valid, most have no data       ValidPairs, a relation, checked before the call
a listed pair can still have no data                    the reply check, after the call
```

The second round:

```
Known + Reveals(SitesTable)   26 callable endpoints, Data/SiteSpecies among them
→ Records, AsList, Clean      SiteSpeciesTable: 650 rows, one per (site, species)
→ zip, set                    ValidPairs: 646 pairs out of 1,548 possible
→ RecordPaths, Follow         the paths we rely on, checked against the API
→ GetReadings                 check the pair → call → check the reply → a clean table of hours
```

Collecting history. `pipeline.py` is the last steps of this notebook, made to run again and again: the same `Get`, `AsList`, `Follow` with `RecordPaths`, `Clean`, and `GetReadings` with both checks, then a save to air.duckdb. Each run fetches the last 7 days of hourly NO₂ for MY1 and GT1. Run `python pipeline.py` first, because this notebook only reads the database. The 7 days overlap from one run to the next, and the table's primary key means each hour is stored once: a new hour is inserted, and an hour we already have is updated. Each run is logged in `runs`.

In [30]:
Database = duckdb.connect("air.duckdb", read_only=True)
Database.sql("SELECT * FROM runs ORDER BY RunAt")

RunAtTIMESTAMP,StartDateDATE,EndDateDATE,FetchedINTEGER,NewINTEGER
2026-11-01 18:15:52.801697,2026-10-26,2026-11-02,326,2
2026-11-01 19:17:18.016359,2026-10-26,2026-11-02,328,2
2026-11-02 11:19:39.033254,2026-10-27,2026-11-03,312,32
2026-11-02 11:29:17.814210,2026-10-27,2026-11-03,312,0
2026-11-02 11:58:35.764301,2026-10-27,2026-11-03,312,0


`changes` records every stored value that a later run changed. So far each one is a late arrival: an hour that was empty when it was first fetched, and filled in by a later run. None is yet a correction of a real value.

In [31]:
Database.sql("SELECT * FROM changes ORDER BY RunAt")

RunAtTIMESTAMP,SiteCodeVARCHAR,SpeciesCodeVARCHAR,MeasurementDateGMTTIMESTAMP,OldValueDOUBLE,NewValueDOUBLE
2026-11-01 12:17:22.575300,GT1,NO2,2026-11-01 11:00:00,NULL,10.6
2026-11-01 17:43:03.790294,MY1,NO2,2026-11-01 12:00:00,NULL,27.2
2026-11-01 17:43:03.790294,MY1,NO2,2026-11-01 11:00:00,NULL,26.7
2026-11-01 17:43:03.790294,GT1,NO2,2026-11-01 12:00:00,NULL,10.7
2026-11-01 18:15:52.786691,GT1,NO2,2026-11-01 17:00:00,NULL,13.5
2026-11-02 11:19:39.016093,MY1,NO2,2026-11-01 17:00:00,NULL,32.4
2026-11-02 11:19:39.016093,GT1,NO2,2026-11-01 18:00:00,NULL,16.2
2026-11-02 11:19:39.016093,MY1,NO2,2026-11-01 18:00:00,NULL,36.8
2026-11-02 11:19:39.016093,GT1,NO2,2026-11-01 19:00:00,NULL,15.3
2026-11-02 11:19:39.016093,MY1,NO2,2026-11-01 19:00:00,NULL,31.9


Shaping. The API sends everything as text, with @ in front of field names and "" for a missing value. `Clean` fixes the names and the missing values, and the `readings` table gives each column a type, a timestamp and a number. Its primary key says what one row is: one site, one species, one hour, and its value. The request window isn't part of a reading, so it lives in `runs`. `sites` is the London site list, refreshed on every run, and joins to `readings` on SiteCode, which has the same name in both.

In [32]:
Database.sql("DESCRIBE readings")

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
SiteCode,VARCHAR,NO,PRI,NULL,NULL
SpeciesCode,VARCHAR,NO,PRI,NULL,NULL
MeasurementDateGMT,TIMESTAMP,NO,PRI,NULL,NULL
Value,DOUBLE,YES,NULL,NULL,NULL


Using it. The question: how much worse is the air right beside a busy road than nearby, and at what time of day? `sites` says MY1, on Marylebone Road, is Kerbside, and GT1, Station Walk, is Urban Background. `HourlyTable` puts them side by side, one row per hour, and `BothTable` keeps only the hours where both sites have a value, so a gap at one site can't skew the comparison. In: `readings` joined to `sites`. Out: a table with one row per hour and a column per site type.

In [33]:
HourlyTable: pandas.DataFrame = Database.sql("""
SELECT MeasurementDateGMT AS Hour,
       avg(Value) FILTER (WHERE SiteType = 'Kerbside')         AS Kerbside,
       avg(Value) FILTER (WHERE SiteType = 'Urban Background') AS Background
FROM readings JOIN sites USING (SiteCode)
WHERE SpeciesCode = 'NO2'
GROUP BY Hour
ORDER BY Hour
""").df()
BothTable: pandas.DataFrame = HourlyTable.dropna()
print(len(BothTable), "of", len(HourlyTable), "hours have a value at both sites")

320 of 396 hours have a value at both sites


Over those hours, the kerbside mean against the background mean. DuckDB can query a DataFrame by its name, so `BothTable` works as a table in SQL.

In [34]:
Database.sql("""
SELECT count(*) AS Hours, round(avg(Kerbside), 1) AS Kerbside, round(avg(Background), 1) AS Background,
       round(avg(Kerbside) / avg(Background), 1) AS Ratio
FROM BothTable
""")

HoursBIGINT,KerbsideDOUBLE,BackgroundDOUBLE,RatioDOUBLE
320,47.6,13.3,3.6


The same by hour of day. Gap is the kerbside mean minus the background mean.

In [35]:
ByHourOfDay: pandas.DataFrame = Database.sql("""
SELECT hour(Hour) AS HourOfDay, round(avg(Kerbside), 1) AS Kerbside, round(avg(Background), 1) AS Background,
       round(avg(Kerbside) - avg(Background), 1) AS Gap
FROM BothTable
GROUP BY HourOfDay
ORDER BY HourOfDay
""").df().set_index("HourOfDay")
ByHourOfDay

,Kerbside,Background,Gap
HourOfDay,,,
0,42.5,11.2,31.3
1,37.8,12.6,25.2
2,36.1,11.1,24.9
3,32.5,8.7,23.8
4,32.3,10.1,22.3
5,38.4,11.8,26.6
6,48.9,14.2,34.7
7,56.9,17.7,39.2
8,55.6,18.5,37.0


As a chart. This also saves no2_by_hour.png for the README.

In [36]:
Chart = ByHourOfDay[["Kerbside", "Background"]].plot(marker="o", title="Mean NO₂ by hour of day: Marylebone Road (kerbside) and Station Walk (background)",
                                                     xlabel="Hour of day (GMT)", ylabel="Mean NO₂ (µg/m³)")
Chart.figure.savefig("no2_by_hour.png", bbox_inches="tight")

Close the database, so that `pipeline.py` can write to it again.

In [37]:
Database.close()